## *Installing required packages*

In [1]:
!pip install -U peft transformers bitsandbytes accelerate trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Importing the libaries

In [2]:
import torch
from transformers import(
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    HfArgumentParser
)
from peft import LoraConfig, PeftModel
from datasets import load_dataset
from trl import SFTTrainer,SFTConfig

In this case of a LLama model we need a prompt template for finetuning chat models



```
# System prompt - to guide the model
User Prompt - to give instruction
Model Answer
```





Generic prompt format for llama chat model

```
# <s>[INST]<<SYS>>
System prompt
<</SYS>>
User Prompt [/INST] </s>
```



we need to reformat our dataset to follow this template
Using this dataset - https://huggingface.co/datasets/timdettmers/openassistant-guanaco repo from hf



note: we dont have to follow a prompt format because we are using a llama base model, instead of chat model

In [3]:
#Loading and Formatting dataset

from datasets import load_dataset
import re

dataset = load_dataset("timdettmers/openassistant-guanaco")

#loading small number of rows since we are running on T4 GPU
dataset = dataset['train'].select(range(1000))

#lets define a function to transform the data
def transform_dataset(batch):
    reformatted_segments = []

    for conversation_text in batch["text"]:
        segments = conversation_text.split("###")
        for i in range(1, len(segments) - 1, 2):
            human_text = segments[i].strip().replace("Human:", "").strip()
            if i + 1 < len(segments):
                bot_text = segments[i + 1].strip().replace("Assistant:", "").strip()
                reformatted_segments.append(f"<s>[INST] {human_text}[/INST] {bot_text}</s>")
            else:
                reformatted_segments.append(f"<s>[INST] {human_text}[/INST]</s>")

    return {"text": reformatted_segments}

transformed_dataset = dataset.map(
    transform_dataset,
    batched=True,
    remove_columns=dataset.column_names,
)
#print(transformed_dataset[0])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/395 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


openassistant_best_replies_train.jsonl:   0%|          | 0.00/20.9M [00:00<?, ?B/s]

openassistant_best_replies_eval.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9846 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/518 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

##  Finetuning Setup

In [4]:
#load the model and train it on given 1000 samples.
model_name = "NousResearch/Hermes-3-Llama-3.1-8B"

# Fine-tuned result model name
#new_model = "Llama-3.1-8b-chat-finetune"

In [5]:
# bitsandbytes parameters
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
#loading model and tokenizer
model = AutoModelForCausalLM.from_pretrained(model_name,quantization_config = bnb_config, device_map = "auto")
model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/883 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

In [6]:
# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

In [7]:
# TrainingArguments parameters, SFT Config
sft_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    fp16=False,
    bf16=False,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    learning_rate=2e-4,
    weight_decay=0.001,
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",
    max_steps=-1,
    report_to="none",
    warmup_ratio=0.03,
    group_by_length=True,
    save_steps=0,
    logging_steps=25,
    max_seq_length=128,
    packing=False,
    dataset_text_field="text"
)

In [9]:
#Setting up SFTtrainer and kicking of training
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=transformed_dataset,
    peft_config=peft_config,
    args=sft_config
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/1263 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1263 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1263 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
25,1.966500
50,1.965200
75,1.568800


Step,Training Loss
25,1.966500
50,1.965200
75,1.568800
100,1.751100
125,1.563500
150,1.688300
175,1.589200
200,1.641400
225,1.487300
250,1.714100


TrainOutput(global_step=316, training_loss=1.6735908532444435, metrics={'train_runtime': 2593.7159, 'train_samples_per_second': 0.487, 'train_steps_per_second': 0.122, 'total_flos': 6644609719934976.0, 'train_loss': 1.6735908532444435})

## Saving model locally, uncomment for uploading to HF

In [10]:
#from huggingface_hub import login, create_repo, upload_folder

#login(token="your_hf_token")

#comment if you want to save only on HF
save_path = "./lora_finetuned_model"
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# repo_name = "your-username/Llama-3.1-8b-chat-finetune"
# create_repo(repo_name, exist_ok=True)

# upload_folder(
#     repo_id=repo_name,
#     folder_path=save_path,
#     path_in_repo=".",
#     commit_message="Upload LoRA-finetuned model"
# )


('./lora_finetuned_model/tokenizer_config.json',
 './lora_finetuned_model/special_tokens_map.json',
 './lora_finetuned_model/chat_template.jinja',
 './lora_finetuned_model/additional_chat_templates/tool_use.jinja',
 './lora_finetuned_model/tokenizer.json')

## Inferencing on this model

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
torch.cuda.empty_cache()

# Path to the fine-tuned model
MODEL_DIR = "./lora_finetuned_model"

# Load tokenizer and base model (with quantization)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
#loading with bnb again since we used bnb for training
base_model = AutoModelForCausalLM.from_pretrained(
    "NousResearch/Hermes-3-Llama-3.1-8B",
    device_map="cuda",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

# Load fine-tuned LoRA adapter, since we used peft for tuning
model = PeftModel.from_pretrained(base_model, MODEL_DIR)
model.eval()

human_input = "What is the distance of earth from the sun?"
formatted_prompt = f"<s>[INST] {human_input} [/INST]"

inputs = tokenizer(human_input, return_tensors="pt").to("cuda")
with torch.no_grad():
    output = model.generate(
      **inputs,
      max_new_tokens=64,
      do_sample=True,
      temperature=0.7,
      top_p=0.9,
      repetition_penalty=1.2,
      eos_token_id=tokenizer.eos_token_id,
      pad_token_id=tokenizer.eos_token_id,
  )

response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

KeyboardInterrupt: 